# Sensei shelf monitor on free Google Colab

Runs the whole pipeline (fit, predict, dashboard) on a free **T4 GPU**. No paid tier, no API keys:
the CLIP and DINOv2 weights are free Hugging Face downloads.

**Before you start**
1. **Runtime > Change runtime type > Hardware accelerator: T4 GPU > Save.** (The notebook is already
   saved with a T4 request; CPU also works, just slower.)
2. Put these files in one Google Drive folder, `MyDrive/sensei/`:
   - `junctionx-sensei.zip`: the project folder, made on your computer with
     `python colab/make_zip.py` (Python zipfile, forward slashes; leaves out `.venv`, `cache`, `outputs`,
     keeps `data/labels/` and `models/`). Do not use Windows "Compress-Archive": its backslash paths break on Linux.
   - `Frutas e Legumes (train & test).zip`: the challenge images
   - `planogram.json` (skip it if it is inside the dataset zip)
   - optional: `validation.zip` (or a folder) with the hidden validation images, and `annotations.json` if you have truth to score against
3. Run the cells top to bottom. Results are zipped back to the same Drive folder at the end.

In [ ]:
# 1. GPU check
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Use Runtime > Change runtime type > T4 GPU, then re-run (CPU works but is slower).")

In [ ]:
# 2. Mount Google Drive (a pop-up asks for permission)
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 3. Settings: edit only if your Drive layout differs
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/sensei")
PROJECT_ZIP = DRIVE_DIR / "junctionx-sensei.zip"
DATASET_ZIP = DRIVE_DIR / "Frutas e Legumes (train & test).zip"
PLANOGRAM_JSON = DRIVE_DIR / "planogram.json"      # optional if the dataset zip contains one
VAL_SOURCE = DRIVE_DIR / "validation.zip"          # optional: a .zip or a folder (<camera>/<ts>.jpg)
ANNOTATIONS_JSON = DRIVE_DIR / "annotations.json"  # optional: only used to score the validation run
WORK = Path("/content")                            # Colab's fast local disk

for required in (PROJECT_ZIP, DATASET_ZIP):
    assert required.exists(), f"Missing {required}. Upload it to Drive (see the top of the notebook)."
print("Found:", sorted(p.name for p in DRIVE_DIR.iterdir()))

In [ ]:
# 4. Unpack the project and move into it
import os
import shutil
import subprocess
import sys
import zipfile


def unzip(zip_path, dest):
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    return Path(dest)


def sh(*args):
    """Run a command inside the project folder and stream its output into the cell."""
    proc = subprocess.Popen([str(a) for a in args], cwd=PROJECT_DIR, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"command failed (exit {proc.returncode}): {' '.join(map(str, args))}")


extracted = unzip(PROJECT_ZIP, WORK / "project_extract")
roots = sorted((p.parent.parent for p in extracted.rglob("sensei/__init__.py")), key=lambda p: len(p.parts))
assert roots, "No sensei/ package inside the project zip."
PROJECT_DIR = roots[0]
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)

In [ ]:
# 5. Install the (light) requirements; Colab already has torch and transformers
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")

In [ ]:
# 6. Dataset -> data/raw/images/<camera_id>/<unix_ts>.jpg and data/raw/planogram.json
from sensei.data import discover_images, load_planogram

DATA_ROOT = PROJECT_DIR / "data" / "raw"
IMAGES_DIR = DATA_ROOT / "images"
PLANOGRAM = DATA_ROOT / "planogram.json"

raw = unzip(DATASET_ZIP, WORK / "dataset_extract")
plano_src = PLANOGRAM_JSON if PLANOGRAM_JSON.exists() else next(raw.rglob("planogram.json"), None)
assert plano_src is not None, "planogram.json not found: put it next to the zips on Drive."
DATA_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy(plano_src, PLANOGRAM)

areas = load_planogram(PLANOGRAM)  # also checks that the JSON loads
records = discover_images(raw, areas)  # finds <camera>/<ts>.jpg at any depth, skips hidden files
assert records, "No images matched a planogram camera id."
shutil.rmtree(IMAGES_DIR, ignore_errors=True)
for r in records:
    dest = IMAGES_DIR / r.camera_id / r.image_name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(r.path, dest)
shutil.rmtree(raw)
print(f"{len(records)} images from {len({r.camera_id for r in records})} of {len(areas)} cameras "
      f"({sum(map(len, areas.values()))} areas) -> {IMAGES_DIR}")

In [ ]:
# 7. Optional validation images (hidden set), same <camera>/<ts>.jpg layout
VAL_DIR = None
if VAL_SOURCE.is_dir():
    VAL_DIR = VAL_SOURCE
elif VAL_SOURCE.is_file():
    VAL_DIR = unzip(VAL_SOURCE, WORK / "val_images")

if VAL_DIR is None:
    print("No validation set found: the validation cell below will be skipped.")
else:
    val_records = discover_images(VAL_DIR, areas)
    print(f"{len(val_records)} validation images, cameras: {sorted({r.camera_id for r in val_records})}")

## Fit, predict, dashboard

`run` fits on every image of the dataset and writes predictions, `details.json` and the static
dashboard to `outputs/`. With a GPU and the Hugging Face weights downloaded it takes a few minutes.
If teacher labels ship in `data/labels/teacher_labels*.json`, they are passed with `--labels`, so the
model gets the calibration heads (the recommended submission model, same as `models/real_v1_cal`).

In [ ]:
# 8. fit + predict + dashboard on the dataset itself
OUT = PROJECT_DIR / "outputs"
MODEL_DIR = OUT / "model"
LABELS = sorted(str(p) for p in (PROJECT_DIR / "data" / "labels").glob("teacher_labels*.json"))
print("teacher labels:", LABELS or "none (unsupervised model only)")
sh(sys.executable, "-m", "sensei", "run", "--data", IMAGES_DIR, "--planogram", PLANOGRAM,
   "--images", IMAGES_DIR, "--out", OUT, *(["--labels", *LABELS] if LABELS else []))

In [ ]:
# 9. Predict on the validation folder with the fitted model (skipped when none was given)
import json

if VAL_DIR is not None:
    VAL_OUT = OUT / "val"
    VAL_OUT.mkdir(parents=True, exist_ok=True)
    if not MODEL_DIR.exists():  # `run` keeps its model elsewhere: fit one explicitly
        sh(sys.executable, "-m", "sensei", "fit", "--data", IMAGES_DIR, "--planogram", PLANOGRAM,
           "--model", MODEL_DIR, *(["--labels", *LABELS] if LABELS else []))
    sh(sys.executable, "-m", "sensei", "predict", "--model", MODEL_DIR, "--images", VAL_DIR,
       "--out", VAL_OUT / "predictions.json", "--details", VAL_OUT / "details.json")
    sh(sys.executable, "-m", "sensei", "dashboard", "--details", VAL_OUT / "details.json",
       "--data", VAL_DIR, "--out", VAL_OUT / "dashboard")

    # the submission must answer every planogram area of every image
    pred = json.loads((VAL_OUT / "predictions.json").read_text(encoding="utf-8"))
    expected = missing = 0
    for r in discover_images(VAL_DIR, areas):
        got = pred.get(r.camera_id, {}).get(r.image_name, {})
        for a in areas[r.camera_id]:
            expected += 1
            missing += a.area_id not in got
    print(f"{expected} areas expected, {missing} missing")
    assert missing == 0, "some areas are missing from predictions.json"
    if ANNOTATIONS_JSON.exists():
        sh(sys.executable, "-m", "sensei", "score", ANNOTATIONS_JSON, VAL_OUT / "predictions.json")

In [ ]:
# 10. Zip the outputs back to Drive (and copy the submission file next to it)
archive = shutil.make_archive(str(DRIVE_DIR / "sensei_outputs"), "zip", root_dir=PROJECT_DIR, base_dir="outputs")
print(f"Saved {archive} ({os.path.getsize(archive) / 1e6:.1f} MB)")
if VAL_DIR is not None:
    shutil.copy(VAL_OUT / "predictions.json", DRIVE_DIR / "predictions.json")
    print("Submission file:", DRIVE_DIR / "predictions.json")

## Notes
- **Open the dashboard:** unzip `sensei_outputs.zip` on your computer and open `outputs/dashboard/index.html`
  (or `outputs/val/dashboard/index.html` for the validation images). It is a static page, no server needed.
- **Free-tier limits:** a session ends after idle time or about 12 hours, and local files in `/content` are lost.
  Everything important is in Drive, so just re-run the cells. Use *Runtime > Disconnect and delete runtime* when done.
- **If pip asks for a restart:** *Runtime > Restart session*, then re-run from cell 3 (Drive stays mounted).
- **No GPU quota left?** Switch the runtime to CPU; the same cells run, only slower.